In [1]:
import requests
import time

In [2]:

def fetch_page(base_url, params=None, delay=1):
    """Fetch a single web page politely. Returns HTML text, or None on failure."""
    headers = {
        "User-Agent": "PoetryResearch (au789885@uni.au.dk)"
    }
    try:
        response = requests.get(base_url, params=params,
                                headers=headers, timeout=10)
        response.raise_for_status()
        time.sleep(delay)          # pause after a successful fetch
        return response.text

    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch {base_url} ({params}): {e}")
        return None
    


In [3]:
poets = fetch_page("https://poets.org/poems", params={"page": 1})
print("poets.org ->", "got", len(poets), "chars" if poets else "nothing")

pf = fetch_page("https://www.poetryfoundation.org/poems/browse",
                params={"sortBy": "prod_PFONE_post_date_desc", "page": 0, "query": ""})
print("poetryfoundation.org ->", "got", len(pf), "chars" if pf else "nothing")

print(poets)

poets.org -> got 74313 chars
poetryfoundation.org -> got 310776 chars
<!DOCTYPE html>
<html lang="en" dir="ltr" prefix="og: https://ogp.me/ns#">
  <head>
    <meta charset="utf-8" />
<meta name="description" content="Poems - Find the best poems by searching our collection of over 10,000 poems by classic and contemporary poets, including Maya Angelou, Emily Dickinson, Robert Frost, Juan Felipe Herrera, Langston Hughes, Sylvia Plath, Edgar Allan Poe, William Shakespeare, Walt Whitman, and more. You can even find poems by occasion, theme, and form." />
<meta name="abstract" content="Poems - The Academy of American Poets is the largest membership-based nonprofit organization fostering an appreciation for contemporary poetry and supporting American poets." />
<meta name="keywords" content="poetry, poets, Poems" />
<link rel="canonical" href="https://poets.org/poems" />
<meta property="og:site_name" content="Poets.org" />
<meta property="og:type" content="Non-Profit" />
<meta property="og:ur

In [ ]:
from bs4 import BeautifulSoup
all_pages = []
poem_links = []
base_url = "https://www.poets.org"
for page_number in range(2):       
    html = fetch_page("https://www.poets.org/poems", params={"page": page_number})
    soup = BeautifulSoup(html, "html.parser")
    links = soup.find_all("a")
    new_links = [
        base_url + a["href"]
        for a in soup.find_all("a", href=True)
        if a["href"].startswith("/poem/")
    ]
    print(f"Collected page {page_number} ({len(poem_links)} chars)")
    poem_links.extend(new_links)



Collected page 0 (0 chars)
Collected page 1 (20 chars)
Collected page 2 (40 chars)
Collected page 3 (60 chars)
Collected page 4 (80 chars)
Collected page 5 (100 chars)
Collected page 6 (120 chars)
Collected page 7 (140 chars)
Collected page 8 (160 chars)
Collected page 9 (180 chars)
Collected page 10 (200 chars)
Collected page 11 (220 chars)
Collected page 12 (240 chars)
Collected page 13 (260 chars)
Collected page 14 (280 chars)
Collected page 15 (300 chars)
Collected page 16 (320 chars)
Collected page 17 (340 chars)
Collected page 18 (360 chars)
Collected page 19 (380 chars)
Collected page 20 (400 chars)
Collected page 21 (420 chars)
Collected page 22 (440 chars)
Collected page 23 (460 chars)
Collected page 24 (480 chars)
Collected page 25 (500 chars)
Collected page 26 (520 chars)
Collected page 27 (540 chars)
Collected page 28 (560 chars)
Collected page 29 (580 chars)
Collected page 30 (600 chars)
Collected page 31 (620 chars)
Collected page 32 (640 chars)
Collected page 33 (660 cha

In [5]:
poems = []

for url in poem_links:
    new_poems = fetch_page(url, params={"page": 0})
    poems.append(new_poems)

In [ ]:
import re

def parse_poems(html):
    soup = BeautifulSoup(html, "html.parser")

    # TITLE
    title_tag = soup.find("h1")
    poem_title = title_tag.get_text(strip=True) if title_tag else None

    # POEM TEXT
    body_tag = soup.find("div", class_="field--body")
    poem_text = body_tag.get_text(strip=True) if body_tag else None

    # ORIGINAL YEAR
    pub_date = soup.find("div", class_="field field--field_date_published")
    pub_text = pub_date.get_text(strip=True) if pub_date else None
    
    # Extract a 4-digit year from the text, then convert to int
    publication_year = 0
    if pub_text:
        match = re.search(r"\d{4}", pub_text)
        if match:
            publication_year = int(match.group())
    
    # AUTHOR
    author_tag = soup.find("a", href=lambda x: x and "/poet/" in x)
    author = author_tag.get_text(strip=True) if author_tag else None
    
    if publication_year is not None and publication_year > 1999:
        return {
            "title": poem_title,
            "text": poem_text,
            "published": publication_year,
            "author": author,
        }
    
parsed = []
for poem in poems:
    new_parsing = parse_poems(poem)
    parsed.append(new_parsing)



In [8]:
print(parsed[700])

{'title': 'Long Island Sound', 'text': "I see it as it looked one afternoonIn August,—by a fresh soft breeze o'erblown.The swiftness of the tide, the light thereon,A far-off sail, white as a crescent moon.The shining waters with pale currents strewn,The quiet fishing-smacks, the Eastern cove,The semi-circle of its dark, green grove.The luminous grasses, and the merry sunIn the grave sky; the sparkle far and wide,Laughter of unseen children, cheerful chirpOf crickets, and low lisp of rippling tide,Light summer clouds fantastical as sleepChanging unnoted while I gazed thereon.All these fair sounds and sights I made my own.", 'published': 2018, 'author': 'Emma Lazarus'}


In [9]:
import uuid
poems = [p for p in parsed if p is not None]

poems_by_id = {f"P{i:04d}": p for i, p in enumerate(poems, start=1000)}

print(poems_by_id["P1000"])

{'title': 'The Summer Day', 'text': 'Who made the world?Who made the swan, and the black bear?Who made the grasshopper?This grasshopper, I mean—the one who has flung herself out of the grass,the one who is eating sugar out of my hand,who is moving her jaws back and forth instead of up and down—who is gazing around with her enormous and complicated eyes.Now she lifts her pale forearms and thoroughly washes her face.Now she snaps her wings open, and floats away.I don’t know exactly what a prayer is.I do know how to pay attention, how to fall downinto the grass, how to kneel down in the grass,how to be idle and blessed, how to stroll through the fields,which is what I have been doing all day.Tell me, what else should I have done?Doesn’t everything die at last, and too soon?Tell me, what is it you plan to dowith your one wild and precious life?', 'published': 2004, 'author': 'Mary Oliver'}


In [ ]:
import pandas as pd
import re

df = pd.DataFrame.from_dict(poems_by_id, orient="index")

for col in df.select_dtypes(include="object").columns:

    df[col] = df[col].str.replace(r'([.!?,;:"])(?=[A-Za-z])', r'\1 ', regex=True)

    df[col] = df[col].str.replace(r'(?<=[a-z])(?=[A-Z])', ' ', regex=True)

df.to_csv("poems6.csv", index=True, encoding="utf-8")

C:\Users\Sipos Dorottya\AppData\Local\Temp\ipykernel_17764\1186039852.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
